# Complex Cell NEURON Notebook Example

This notebook loads an SWC morphology, inserts Hodgkin-Huxley channels, and displays voltage on the morphology with a selected-segment voltage trace. The source is built inside the notebook backend process through `cnv.show(build_source)`, so live NEURON objects are not captured by the notebook kernel.


In [1]:
import compneurovis as cnv


In [ ]:
DT = 0.25
DISPLAY_DT = 0.25
TRACE_WINDOW_MS = 160.0
DENDRITE_NSEG = 10


def build_source():
    import pathlib

    from neuron import h

    import compneurovis as cnv
    from compneurovis.backends.neuron.io import load_swc_neuron

    repo = pathlib.Path(cnv.__file__).resolve().parents[2]
    swc_path = repo / "res" / "Animal_2_Basal_2.CNG.swc"
    sections = load_swc_neuron(str(swc_path))

    for sec in sections:
        sec.insert("hh")
        if "soma" not in sec.name().lower():
            sec.nseg = DENDRITE_NSEG

    soma = next(sec for sec in sections if "soma" in sec.name().lower())
    stim_scale = {"value": 0.75}
    clamp_specs = []
    for delay, dur, base_amp in (
        (2.0, 5.0, 1.0),
        (24.0, 5.0, 0.8),
        (48.0, 5.0, 1.0),
        (72.0, 5.0, 0.8),
        (96.0, 5.0, 1.0),
    ):
        clamp = h.IClamp(soma(0.5))
        clamp.delay = delay
        clamp.dur = dur
        clamp.amp = stim_scale["value"] * base_amp
        clamp_specs.append((clamp, base_amp))

    def set_stim_scale(ctx, value):
        stim_scale["value"] = float(value)
        for clamp, base_amp in clamp_specs:
            clamp.amp = stim_scale["value"] * base_amp

    h.dt = DT
    h.celsius = 6.3
    h.finitialize(-65.0)

    src = cnv.neuron.source(
        sections=sections,
        dt=DT,
        display_dt=DISPLAY_DT,
        flush_dt=0.5,
        v_init=-65.0,
        title="Complex cell notebook",
    )
    morph = src.morphology(
        variable="v",
        name="Voltage morphology",
        unit="mV",
        color_limits=(-80.0, 50.0),
        selected=f"{soma.name()}@0.50000",
        max_refresh_hz=60.0,
    )
    src.line(
        "Selected voltage",
        source=morph.selection,
        rolling_window=TRACE_WINDOW_MS,
        y_label="Voltage",
        y_unit="mV",
        y_min=-85.0,
        y_max=55.0,
        color="#00d2be",
        max_refresh_hz=15.0,
    )

    current_data = src.record_refs(
        "Soma input current",
        refs=tuple(clamp._ref_i for clamp, _ in clamp_specs),
        series=tuple(f"Pulse {index + 1}" for index in range(len(clamp_specs))),
        unit="nA",
        window=TRACE_WINDOW_MS,
    )
    src.line(
        "Soma input current",
        source=current_data,
        rolling_window=TRACE_WINDOW_MS,
        y_label="Current",
        y_unit="nA",
        y_min=-0.1,
        y_max=1.2,
        color="#2356b8",
        max_refresh_hz=15.0,
    )

    src.slider(
        "stim_scale",
        label="Stimulus scale",
        get=lambda: stim_scale["value"],
        set=set_stim_scale,
        min=0.0,
        max=1.5,
        steps=150,
    )
    src.button("reset", label="Reset", fn=lambda ctx: ctx.reset())

    return src


In [3]:
widget = cnv.show(build_source)
widget
